# 02 — SCF feature column discovery

Load the 2022 SCF summary extract and **list column names** that may support financial profiling, grouped by theme. Matching uses **case-insensitive** substring or word-boundary rules on variable names only — **no preprocessing** of the data.

Data path: `datasets/raw/scfp2022s/rscfp2022.dta`

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for directory in (cwd, *cwd.parents):
        if (directory / "backend").is_dir() and (directory / "ml").is_dir():
            return directory
    return cwd


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "datasets" / "raw" / "scfp2022s" / "rscfp2022.dta"

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"SCF extract not found at {DATA_PATH}. "
        "Place rscfp2022.dta under datasets/raw/scfp2022s/."
    )

df = pd.read_stata(DATA_PATH)
print(f"Loaded: {DATA_PATH}  |  shape: {df.shape}")

## Helpers: name matching

- **Word** = token surrounded by non-alphanumeric boundaries (avoids `wage` → `age`).
- **Substring** = literal needle in the lowered column name (for multi-letter keywords).

In [ ]:
COLUMNS = [str(c) for c in df.columns]


def lowered(name: str) -> str:
    return name.lower()


def has_word(name: str, word: str) -> bool:
    return bool(re.search(rf"(?i)(?<![a-z0-9]){re.escape(word)}(?![a-z0-9])", name))


def has_any_word(name: str, words: tuple[str, ...]) -> bool:
    return any(has_word(name, w) for w in words)


def has_any_substr(name: str, needles: tuple[str, ...]) -> bool:
    s = lowered(name)
    return any(n in s for n in needles)


def pick_columns(predicate) -> list[str]:
    return sorted(c for c in COLUMNS if predicate(c))

## Thematic groups (keyword-inspired)

Rules are intentionally broad for discovery; overlap across groups is allowed.

In [ ]:
THEMATIC: dict[str, list[str]] = {
    "Age": pick_columns(
        lambda c: has_word(c, "age")
    ),
    "Income-related": pick_columns(
        lambda c: has_any_substr(
            c,
            ("income", "wage", "salary", "payroll", "earnings", "earn"),
        )
    ),
    "Net worth": pick_columns(
        lambda c: has_any_substr(c, ("worth", "networth", "net_worth", "wealth"))
        or (has_word(c, "net") and has_any_substr(c, ("worth", "wealth")))
    ),
    "Asset-related": pick_columns(
        lambda c: has_any_substr(c, ("asset", "home", "house", "property", "equity"))
    ),
    "Debt-related": pick_columns(
        lambda c: has_any_substr(
            c,
            ("debt", "mortgage", "loan", "credit", "borrow", "owe", "lien"),
        )
    ),
    "Family structure": pick_columns(
        lambda c: has_any_substr(
            c,
            (
                "family",
                "child",
                "spouse",
                "marital",
                "depend",
                "household",
                "member",
                "fertility",
                "married",
                "single",
                "divorc",
                "widow",
            ),
        )
    ),
    "Employment-related": pick_columns(
        lambda c: has_any_substr(
            c,
            ("job", "employ", "occup", "worker", "unemp", "selfemp", "business", "firm"),
        )
        or has_any_word(c, ("work",))
    ),
}

for title, cols in THEMATIC.items():
    display(Markdown(f"### {title} ({len(cols)} columns)"))
    display(pd.Series(cols, name="column", dtype="string"))

## Raw keyword sweep (user-listed patterns)

Each keyword lists columns whose **name** contains that substring (case-insensitive).  
Exception: **`net`** uses a word-boundary match or `networth` / `net_worth` style tokens to reduce noise.

In [ ]:
KEYWORDS = [
    "income",
    "asset",
    "debt",
    "worth",
    "home",
    "house",
    "mortgage",
    "loan",
    "credit",
    "job",
    "employ",
    "net",
]


def matches_keyword(column: str, keyword: str) -> bool:
    if keyword == "net":
        s = lowered(column).replace(" ", "")
        if "networth" in s or "net_worth" in lowered(column):
            return True
        return has_word(column, "net")
    return keyword in lowered(column)


KEYWORD_GROUPS: dict[str, list[str]] = {
    kw: sorted(c for c in COLUMNS if matches_keyword(c, kw)) for kw in KEYWORDS
}

for kw, cols in KEYWORD_GROUPS.items():
    display(Markdown(f"### Keyword `{kw}` ({len(cols)} columns)"))
    display(pd.Series(cols, name="column", dtype="string"))

## Overlap summary

How many columns each thematic bucket has (a column may appear in more than one bucket).

In [ ]:
summary = (
    pd.DataFrame(
        {"column_count": {k: len(v) for k, v in THEMATIC.items()}},
    )
    .sort_values("column_count", ascending=False)
)
summary